In [1]:
#####
#####This file extracts all code and code metrics of a project AZURE ######
#

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import subprocess 
import lizard
import pandas as pd
from config import AZURE_TOKEN, AZURE_REPO_NAME, AZURE_PROJECT, AZURE_ORG, NAME_FILE_METRICS_CODE, SUPPORTED_EXTENSIONS


REPO_URL = f'https://{AZURE_TOKEN}@dev.azure.com/{AZURE_ORG}/{AZURE_PROJECT}/_git/{AZURE_REPO_NAME}'

#path where the clone project will be saved
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
# if folder exist empty--> better delete folder and clone again
clone_dir = os.path.join(parent_dir, 'target_repo')

#create the folder if it does not exist
if not os.path.exists(clone_dir):
    os.makedirs(clone_dir)
    print(f"Folder '{clone_dir}' created.")

# Clone the repository if not already cloned
if not os.listdir(clone_dir):
    print(f"Cloning from: {REPO_URL}")
    print(f"Cloning to: {clone_dir}")
    try:
        result = subprocess.run(['git', 'clone', REPO_URL, clone_dir], check=True, capture_output=True, text=True)
        print("Repository cloned successfully.")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error cloning the repository: {e}")
        print(e.stderr)
    except Exception as e:
        print(f"Unexpected error cloning the repository: {e}")
else:
    print(f"The folder '{clone_dir}' already exists and this is not empty. It won't be cloned.")

# pull
if os.path.exists(clone_dir) and os.listdir(clone_dir):
    try:
        print(f"Doing pull in: {clone_dir}")
        result = subprocess.run(["git", "-C", clone_dir, "pull"], check=True, capture_output=True, text=True)
        print("Repository updated successfully.")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error doing the pull of repository: {e}")
        print(e.stderr)
    except Exception as e:
        print(f"Unexpected Error doing the pull: {e}")
else:
    print("The repository could not be cloned, so no pull will be attempted.")

    
#verify if the extension is in the list   
def get_all_files(repo_path, extensions=SUPPORTED_EXTENSIONS):
    files = []
    for root, dirs, file_names in os.walk(repo_path):
        for file_name in file_names:
            if any(file_name.endswith(ext) for ext in extensions):
                files.append(os.path.join(root, file_name))
    return files

# Function for getting code from a file
def get_source_code(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()
    except Exception as e:
        print(f"Error reading the file {file_path}: {e}")
        return ""


def calculate_file_metrics(file_path):
    analysis = lizard.analyze_file(file_path)
    function_count = len(analysis.function_list)
    token_count = sum(func.token_count for func in analysis.function_list)
    parameter_count = sum(func.parameter_count for func in analysis.function_list)
    nloc = sum(func.nloc for func in analysis.function_list)    
    cyclomatic_complexity = sum(func.cyclomatic_complexity for func in analysis.function_list)      
    density = round(cyclomatic_complexity / nloc, 2) if nloc > 0 else 0    
    length = sum(func.length for func in analysis.function_list)

    if cyclomatic_complexity <= 10:
        complexity_rank = 'low'
    elif cyclomatic_complexity <= 20:
        complexity_rank = 'moderate'
    elif cyclomatic_complexity <= 50:
        complexity_rank = 'high'
    else:
        complexity_rank = 'very high'
    
    relative_path = os.path.relpath(file_path, clone_dir).replace('\\', '/')
    
    return {
        "file_name": os.path.basename(file_path),
        "nloc": nloc,
        "cyclomatic_complexity": cyclomatic_complexity,
        "density": density,
        "complexity_rank": complexity_rank,
        "token_count": token_count,
        "parameter_count": parameter_count,
        "length": length,
        "function_count": function_count,
        "location": f"/{relative_path}"              
    }

 
# Initialize DataFrame for file metrics
metrics_data = []
files = get_all_files(clone_dir)
print(f"Quantity of files found: {len(files)}")

for file_path in files:
    file_metrics = calculate_file_metrics(file_path)
    metrics_data.append(file_metrics)


# dataframes
metrics_df = pd.DataFrame(metrics_data)

# cleaning data, specify what columns you want to remove if they have None, NaN
metrics_df.dropna(subset=['file_name'], inplace=True)

# Save data in CSV files inside the 'data' folder
data_folder = os.path.join(os.getcwd(), '..', 'data', 'extractions')

os.makedirs(data_folder, exist_ok=True)

metrics_df.to_csv(os.path.join(data_folder, NAME_FILE_METRICS_CODE), index=False)

print(f"Data metrics code successfully extracted and saved in '{NAME_FILE_METRICS_CODE}'")

The folder 'c:\Users\B552LX\OneDrive - AXA\Documentos\SDP\software-defect-prediction\target_repo' already exists and this is not empty. It won't be cloned.
Doing pull in: c:\Users\B552LX\OneDrive - AXA\Documentos\SDP\software-defect-prediction\target_repo
Error doing the pull of repository: Command '['git', '-C', 'c:\\Users\\B552LX\\OneDrive - AXA\\Documentos\\SDP\\software-defect-prediction\\target_repo', 'pull']' returned non-zero exit status 1.
error: cannot stat 'referentielcontrat-application/referentielcontrat-application-prevoyance/src/main/java/fr/axa/eip/prspcrefcnt/col/controllers/HistoriqueContratCompositionPrevoyanceController.java': Filename too long
error: cannot stat 'referentielcontrat-application/referentielcontrat-application-prevoyance/src/main/java/fr/axa/eip/prspcrefcnt/col/controllers/HistoriqueSectionCompositionPrevoyanceController.java': Filename too long
error: cannot stat 'referentielcontrat-application/referentielcontrat-application-prevoyance/src/test/java/f